In [0]:
from pyspark.sql.types import StructType, StringType
from pyspark.sql.functions import current_timestamp, col

In [0]:
file_path_transacciones = "/Volumes/workspace/finanzas/datasets/Transacciones.csv"
file_path_clientes = "/Volumes/workspace/finanzas/datasets/Clientes.json"

In [0]:
transacciones = spark.read.csv(file_path_transacciones, sep="|", header=True, inferSchema=False)

transacciones = transacciones.withColumnRenamed("id_usuario", "id_cliente") \
                                .withColumn("fecha_carga", current_timestamp())

In [0]:
schema = (
          StructType().add("Celular", StringType())
                      .add("Correo", StringType())
                      .add("Documento", StringType())
                      .add("FechaInscripcion", StringType())
                      .add("Representante", StringType())
                      .add("id_usuario", StringType())
        )

clientes = (
            spark.read.format("json")
                      .option("multiline", "true")
                      .schema(schema)
                      .load(file_path_clientes)
            )

clientes = (
            clientes.select(
                            col("id_usuario").alias("id_cliente"),
                            col("Documento").alias("documento"),
                            col("Representante").alias("representante"),
                            col("Celular").alias("celular"),
                            col("Correo").alias("correo"),
                            col("FechaInscripcion").alias("fecha_inscripcion")
                          ).withColumn("fecha_carga", current_timestamp())
          )

In [0]:
try:
    transacciones.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("finanzas.bronze_transacciones")

    clientes.write.format("delta").mode("overwrite").option("overwriteSchema", True).saveAsTable("finanzas.bronze_clientes")

except Exception as e:
    import traceback
    # Tipo de error
    error_type = type(e).__name__
    # Descripcion del error
    error_summary = str(e)
    # Traza del error (ver en que parte se generó el error)
    error_trace = traceback.format_exc()
    
    # Error completo
    error_msg_full = f"{error_type}: {error_summary}\n{error_trace}"

    if len(error_msg_full) > 500:
        error_msg = error_msg_full[:500] + "\n[...] ERROR TRUNCADO [...]"
    else:
        error_msg = error_msg_full

    dbutils.jobs.taskValues.set(key="error", value=error_msg)
    raise e